# 01 — Data cleaning

**Does:** cleans 5 years of daily state-level demand data (Jan 2020 – Mar 2025).

**Input:** `data/raw/india_electricity_data.csv`

**Output:** `data/processed/india_electricity_data_updated.csv`

**Steps:** drop Essar Steel and Jharkhand → merge DD + DNH → forward/backward fill nulls within each state.

In [22]:
import pandas as pd


In [23]:
df=pd.read_csv("../data/raw/india_electricity_data.csv")


In [24]:
df.head()

,Date,State,Max Demand Met,Shortage During Peak,Energy Met,Drawl Schedule,OD(+) / UD(-),Max OD,Energy Shortage
0,2020-01-01 00:00:00,Andhra Pradesh,8561.0,0.0,170.1,66.3,-0.5,367.0,0.0
1,2020-01-01 00:00:00,Arunachal Pradesh,109.0,1.0,2.2,1.9,0.2,34.0,0.0
2,2020-01-01 00:00:00,Assam,1358.0,28.0,23.4,17.3,1.3,90.0,0.2
3,2020-01-01 00:00:00,Bihar,4656.0,0.0,82.0,81.2,-0.7,255.0,0.0
4,2020-01-01 00:00:00,Chandigarh,306.0,0.0,4.9,4.7,0.2,46.0,0.0


In [25]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 69012 entries, 0 to 69011
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Date                  69012 non-null  str    
 1   State                 69012 non-null  str    
 2   Max Demand Met        62972 non-null  float64
 3   Shortage During Peak  64006 non-null  float64
 4   Energy Met            64014 non-null  float64
 5   Drawl Schedule        64024 non-null  float64
 6   OD(+) / UD(-)         64024 non-null  float64
 7   Max OD                64020 non-null  float64
 8   Energy Shortage       64024 non-null  float64
dtypes: float64(7), str(2)
memory usage: 6.5 MB


In [26]:
df.isnull().sum() # is find how many null values are present in the dataset

Date                       0
State                      0
Max Demand Met          6040
Shortage During Peak    5006
Energy Met              4998
Drawl Schedule          4988
OD(+) / UD(-)           4988
Max OD                  4992
Energy Shortage         4988
dtype: int64

Checking the total missing values for each state.

In [27]:
df[['Max Demand Met', 'Energy Met', 'Shortage During Peak', 'OD(+) / UD(-)', 'Energy Shortage', 'Max OD']].isnull().groupby(df['State']).sum()

,Max Demand Met,Energy Met,Shortage During Peak,OD(+) / UD(-),Energy Shortage,Max OD
State,,,,,,
Andhra Pradesh,43,43,43,43,43,43
Arunachal Pradesh,43,43,43,43,43,43
Assam,29,29,29,29,29,29
Bihar,29,29,29,29,29,29
Chandigarh,29,29,29,29,29,29
Chhattisgarh,30,30,30,30,30,30
DD,1056,1056,1056,1053,1053,1053
DNH,1056,29,29,29,29,29
DVC,29,29,29,29,29,29


In [28]:
df_updated = df[~df['State'].isin(['Essar steel', 'Jharkhand'])].copy()

Exclusions Jharkhand, Essar.

In [29]:
df_updated[['Max Demand Met', 'Energy Met', 'Shortage During Peak', 'OD(+) / UD(-)', 'Energy Shortage', 'Max OD']].isnull().groupby(df_updated['State']).sum()

,Max Demand Met,Energy Met,Shortage During Peak,OD(+) / UD(-),Energy Shortage,Max OD
State,,,,,,
Andhra Pradesh,43,43,43,43,43,43
Arunachal Pradesh,43,43,43,43,43,43
Assam,29,29,29,29,29,29
Bihar,29,29,29,29,29,29
Chandigarh,29,29,29,29,29,29
Chhattisgarh,30,30,30,30,30,30
DD,1056,1056,1056,1053,1053,1053
DNH,1056,29,29,29,29,29
DVC,29,29,29,29,29,29


Adding DD and DNH

In [31]:
dd_dnh_states = ['DD', 'DNH']

dd_dnh_daily = (
    df_updated[df_updated['State'].isin(dd_dnh_states)]
    .groupby('Date', as_index=False)
    .sum(numeric_only=True, min_count=1)
)
dd_dnh_daily['State'] = 'DD & DNH'

dd_dnh_daily = dd_dnh_daily[
    ['Date', 'State']
    + [col for col in df_updated.columns if col not in ['Date', 'State']]
]

df_updated = pd.concat(
    [df_updated[~df_updated['State'].isin(dd_dnh_states)], dd_dnh_daily],
    ignore_index=True,
).sort_values(['Date', 'State'], ignore_index=True)

In [12]:
df_updated[['Max Demand Met', 'Energy Met', 'Shortage During Peak', 'OD(+) / UD(-)', 'Energy Shortage', 'Max OD']].isnull().groupby(df_updated['State']).sum()

,Max Demand Met,Energy Met,Shortage During Peak,OD(+) / UD(-),Energy Shortage,Max OD
State,,,,,,
Andhra Pradesh,43,43,43,43,43,43
Arunachal Pradesh,43,43,43,43,43,43
Assam,29,29,29,29,29,29
Bihar,29,29,29,29,29,29
Chandigarh,29,29,29,29,29,29
Chhattisgarh,30,30,30,30,30,30
DD & DNH,1056,29,29,26,26,26
DVC,29,29,29,29,29,29
Delhi,29,29,29,29,29,30


In [13]:
numeric_cols = ['Max Demand Met', 'Shortage During Peak', 'Energy Met', 
                 'Drawl Schedule', 'OD(+) / UD(-)', 'Max OD', 'Energy Shortage']

df_updated = df_updated.sort_values(['State', 'Date']).reset_index(drop=True)
df_updated[numeric_cols] = df_updated.groupby('State')[numeric_cols].ffill()

In [14]:
df_updated[['Max Demand Met', 'Energy Met', 'Shortage During Peak', 'OD(+) / UD(-)', 'Energy Shortage', 'Max OD']].isnull().groupby(df_updated['State']).sum()

,Max Demand Met,Energy Met,Shortage During Peak,OD(+) / UD(-),Energy Shortage,Max OD
State,,,,,,
Andhra Pradesh,0,0,0,0,0,0
Arunachal Pradesh,0,0,0,0,0,0
Assam,0,0,0,0,0,0
Bihar,0,0,0,0,0,0
Chandigarh,0,0,0,0,0,0
Chhattisgarh,0,0,0,0,0,0
DD & DNH,0,0,0,0,0,0
DVC,0,0,0,0,0,0
Delhi,0,0,0,0,0,0


In [32]:
numeric_cols = ['Max Demand Met', 'Shortage During Peak', 'Energy Met', 
                 'Drawl Schedule', 'OD(+) / UD(-)', 'Max OD', 'Energy Shortage']

df_updated = df_updated.sort_values(['State', 'Date']).reset_index(drop=True)
df_updated[numeric_cols] = df_updated.groupby('State')[numeric_cols].bfill()

In [33]:
df_updated[['Max Demand Met', 'Energy Met', 'Shortage During Peak', 'OD(+) / UD(-)', 'Energy Shortage', 'Max OD']].isnull().groupby(df_updated['State']).sum()

,Max Demand Met,Energy Met,Shortage During Peak,OD(+) / UD(-),Energy Shortage,Max OD
State,,,,,,
Andhra Pradesh,0,0,0,0,0,0
Arunachal Pradesh,0,0,0,0,0,0
Assam,0,0,0,0,0,0
Bihar,0,0,0,0,0,0
Chandigarh,0,0,0,0,0,0
Chhattisgarh,0,0,0,0,0,0
DD & DNH,1035,0,0,0,0,0
DVC,0,0,0,0,0,0
Delhi,0,0,0,0,0,0


In [17]:
df_updated.isnull().sum()

Date                    0
State                   0
Max Demand Met          0
Shortage During Peak    0
Energy Met              0
Drawl Schedule          0
OD(+) / UD(-)           0
Max OD                  0
Energy Shortage         0
dtype: int64

In [18]:
df_updated.to_csv('../data/processed/india_electricity_data_updated.csv', index=False)

In [19]:
time = pd.date_range(start="2021-01-01", end="2023-01-01", freq="D")
time.dayofyear

Index([  1,   2,   3,   4,   5,   6,   7,   8,   9,  10,
       ...
       357, 358, 359, 360, 361, 362, 363, 364, 365,   1],
      dtype='int32', length=731)

In [34]:
time

DatetimeIndex(['2021-01-01', '2021-01-02', '2021-01-03', '2021-01-04',
               '2021-01-05', '2021-01-06', '2021-01-07', '2021-01-08',
               '2021-01-09', '2021-01-10',
               ...
               '2022-12-23', '2022-12-24', '2022-12-25', '2022-12-26',
               '2022-12-27', '2022-12-28', '2022-12-29', '2022-12-30',
               '2022-12-31', '2023-01-01'],
              dtype='datetime64[us]', length=731, freq='D')

In [40]:
generator_set = pd.read_csv("../data/raw/india_generator_stack.csv")
generator_set.head()

,State,Fuel_Type,Capacity_MW,Marginal_Cost_Rs_per_MWh
0,Andhra Pradesh,Wind,4461.58,0.0
1,Andhra Pradesh,Solar,8144.32,0.0
2,Andhra Pradesh,Large Hydro,3290.00,1000.0
3,Andhra Pradesh,Small Hydro,164.51,1000.0
4,Andhra Pradesh,Coal,13890.00,2000.0
